In [ ]:
!git clone https://github.com/dave123981/commerce-recommendation-engine.git
%cd commerce-recommendation-engine
!pip install -r requirements.txt -q

In [2]:
!pwd

!ls

/content/commerce-recommendation-engine
data  notebooks  README.md  requirements.txt  src


In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -q kagglehub
import kagglehub
kagglehub.login()   # small browser popup, only needed once per session

!python data/download_data.py          # kagglehub is now the default method
!python data/build_interactions.py

In [ ]:
import pandas as pd

interactions = pd.read_csv("data/interactions.csv", parse_dates=["timestamp"])

print(interactions.shape)
print(interactions.nunique())
print("sparsity:", 1 - len(interactions) / (interactions.user_id.nunique() * interactions.item_id.nunique()))
print("orders per user:", interactions.groupby("user_id").order_id.nunique().describe())
print("purchases per item:", interactions.groupby("item_id").order_id.nunique().describe())

In [ ]:
import data.build_interactions as bi
interactions = bi.build_interactions(min_orders_per_user=10, min_purchases_per_item=50)
interactions.to_csv("data/interactions.csv", index=False)

In [ ]:
from src.metrics import time_based_split

train, test = time_based_split(interactions, test_frac=0.2)
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print(len(train), len(test))

In [ ]:
from src.v1_popularity import PopularityRecommender
from src.metrics import evaluate_model

model = PopularityRecommender(half_life_days=30).fit(train)
metrics = evaluate_model(model, test, k=10)
print(metrics)

In [ ]:
import csv
from pathlib import Path

results_path = Path("results.csv")
write_header = not results_path.exists()
with open(results_path, "a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["version", "precision@10", "recall@10", "map@10", "ndcg@10"])
    writer.writerow(["v1_popularity", metrics["precision@10"], metrics["recall@10"], metrics["map@10"], metrics["ndcg@10"]])

In [ ]:
#model check against random sampling
users_who_bought_top_item = set(train[train.item_id == top_item].user_id)
eligible_users = set(test.user_id) - users_who_bought_top_item

eligible_test = test[test.user_id.isin(eligible_users)]
hit_rate_eligible = eligible_test.groupby("user_id").item_id.apply(lambda x: top_item in set(x)).mean()
print(f"eligible users: {len(eligible_users)} | hit rate among them: {hit_rate_eligible:.4f}")